# Composing local operators and using AMG

A local problem exposes its operator, signed trace coupling, source, kernel and physical mean constraints. The global coupling is independent of how these matrices were assembled. CPU AMG acts on a projected and pinned SPD local complement, not directly on the global saddle matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, solve_darcy, solve_brinkman, solve_elasticity


In [ ]:
from pymhm import LocalProblem, HybridSystem
local=LocalProblem([[1.,-1.],[-1.,1.]],np.eye(2),[0.,0.],np.array([0,1]),kernel=np.ones((2,1)),constraints=np.full((2,1),.5))
system=HybridSystem([local],boundary_load=[0.,1.])
result=system.solve()
np.testing.assert_allclose(result.fields[0],[0.,1.],atol=1e-12)
lu=local.condense('scipy')
amg=local.condense('pyamg')
np.testing.assert_allclose(lu.lifts,amg.lifts,atol=1e-12)
mesh=TriangleMesh.unit_square(2)
solution=solve_darcy(mesh,dirichlet=lambda x:1+x[:,0],local_solver='pyamg')
assert solution.l2_error(lambda x:1+x[:,0])<1e-9
print('AMG MHM pressure error',solution.l2_error(lambda x:1+x[:,0]))


GPU AMG uses the optional AmgX adapter. CPU/GPU backends must meet the same residual criterion, and transfer/setup costs belong in timing. No novelty claim about the first use of AMG with MHM is made.